# 🌊 Offshore Wind LCOE Calculator

**Algorithmic LCOE (Levelized Cost of Energy) Calculation for European Offshore Wind Farms**

Based on `windTurbineData_enriched_2026EUR.csv` data.

**Modules:**
1. **CAPEX** (inflation adjustment and grid connection model)
2. **Energy Yield** (CF from data or estimation by wind_speed)
3. **OPEX** (base + adjustment by distance and foundation type)
4. **LCOE** (discounting, CRF)
5. **Validation and sensitivity analysis**
6. **Project map** (Folium)

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

print("Libraries downloaded.")

Libraries downloaded.


## 1. Data loading

In [6]:
df = pd.read_csv('output/windTurbineData_enriched_2026EUR.csv')

print(f"Dataset size: {df.shape[0]} projects, {df.shape[1]} features")

key_cols = ['wind_farm_name', 'country', 'commissioning_year', 'installed_capacity_MW',
            'foundation_type', 'water_depth_m', 'distance_from_shore_km',
            'total_budget_EUR_2026', 'mean_hub_wind_speed', 'capacity_factor',
            'grid_connection_model', 'project_lifetime_years']
print(df[key_cols].dtypes)

Dataset size: 182 projects, 24 features
wind_farm_name                str
country                       str
commissioning_year        float64
installed_capacity_MW     float64
foundation_type               str
water_depth_m             float64
distance_from_shore_km    float64
total_budget_EUR_2026     float64
mean_hub_wind_speed       float64
capacity_factor               str
grid_connection_model         str
project_lifetime_years    float64
dtype: object


# 2. Data Preprocessing and Cleaning

In [7]:
# 1. Convert capacity_factor to numeric
df['capacity_factor'] = pd.to_numeric(df['capacity_factor'], errors='coerce')

# 2. Fill missing project_lifetime_years (default: 25 years)
df['project_lifetime_years'] = df['project_lifetime_years'].fillna(25.0)

# 3. Classify foundations: Floating vs. Fixed
floating_keywords = ['floating', 'spar', 'semi-submersible', 'tlp', 'damping', 'sath']
df['is_floating'] = df['foundation_type'].str.lower().str.contains(
    '|'.join(floating_keywords), na=False
)
df['foundation_category'] = df['is_floating'].map({True: 'Floating', False: 'Fixed'})

# 4. Filter projects with known budget and capacity
df_model = df[df['total_budget_EUR_2026'].notna() & df['installed_capacity_MW'].notna()].copy()

print(f"✅ Projects after filtering: {len(df_model)}")
print(f"   Floating: {df_model['is_floating'].sum()}, Fixed: {(~df_model['is_floating']).sum()}")
print(f"   Missing CF values: {df_model['capacity_factor'].isna().sum()}/{len(df_model)}")
print(f"   Missing lifetime values: {df_model['project_lifetime_years'].isna().sum()}")

✅ Projects after filtering: 176
   Floating: 14, Fixed: 162
   Missing CF values: 157/176
   Missing lifetime values: 0
